# `TRG_LOC_INSERT_FULL_LOAD.txt` Conversion

**Conversion Timestamp:** 2024-07-30 12:00:00

**Description:** This notebook performs a full load insert into the `TRG_LOC` table from the `LOCATIONS` source table.


In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type")
dbutils.widgets.text("ETL_PROC_WID", "-1", "ETL Process Widget ID")
dbutils.widgets.text("DATASOURCE_NUM_ID", "-1", "Datasource Number ID")
dbutils.widgets.text("ODI_SESS_NO", "-1", "ODI Session Number")
dbutils.widgets.text("ETL_LAST_EXTRACT_TIME", "1900-01-01 00:00:00", "ETL Last Extract Time")
dbutils.widgets.text("ETL_CURRENT_EXTRACT_TIME", "1900-01-01 00:00:00", "ETL Current Extract Time")

# ETL Parameters

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_parameters AS
SELECT
  '${ETL_JOB_TYPE}' AS etl_job_type,
  CAST('${ETL_PROC_WID}' AS BIGINT) AS etl_proc_wid,
  CAST('${DATASOURCE_NUM_ID}' AS BIGINT) AS datasource_num_id,
  CAST('${ODI_SESS_NO}' AS BIGINT) AS odi_sess_no,
  to_timestamp('${ETL_LAST_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_last_extract_time,
  to_timestamp('${ETL_CURRENT_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_current_extract_time;

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_last_extract_time AS
SELECT etl_last_extract_time FROM v_etl_parameters;

CREATE OR REPLACE TEMPORARY VIEW v_etl_current_extract_time AS
SELECT etl_current_extract_time FROM v_etl_parameters;

In [ ]:
display(spark.sql("SELECT * FROM v_etl_parameters"))

# Target Table

In [ ]:
%sql
-- SCEN_TASK_NO in {10}: Placeholder for session start/variable declaration
-- SCEN_TASK_NO in {20}: Another placeholder

-- Create target table if it does not exist
CREATE TABLE IF NOT EXISTS workspace.hr.trg_loc (
  LOCATION_ID      BIGINT,
  STREET_ADDRESS   STRING,
  POSTAL_CODE      STRING,
  CITY             STRING,
  STATE_PROVINCE   STRING,
  COUNTRY_ID       STRING
) USING DELTA;

# Load into Target

In [ ]:
%sql
-- SCEN_TASK_NO in {30}: Insert into target table
INSERT INTO workspace.hr.trg_loc
  (
    LOCATION_ID ,
    STREET_ADDRESS ,
    POSTAL_CODE ,
    CITY ,
    STATE_PROVINCE ,
    COUNTRY_ID 
  ) 
SELECT 
  locations.LOCATION_ID ,
  locations.STREET_ADDRESS ,
  locations.POSTAL_CODE ,
  locations.CITY ,
  locations.STATE_PROVINCE ,
  locations.COUNTRY_ID  
FROM 
  workspace.hr.locations AS locations;

# Optimize Target

In [ ]:
%sql
-- Disable ZORDER stats check to prevent DELTA_ZORDERING_ON_COLUMN_WITHOUT_STATS
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.hr.trg_loc ZORDER BY (LOCATION_ID);

# Cleanup

In [ ]:
%sql
-- No temporary staging or flow tables were used, so no cleanup needed.

# Validation

In [ ]:
%sql
SELECT COUNT(*) AS total_records FROM workspace.hr.trg_loc;

In [ ]:
%sql
SELECT * FROM workspace.hr.trg_loc LIMIT 10;

# Conversion Notes